# Desk capture: TDGSA

The lab-desk simulator saved the two photodiode traces it probed. This runs the same time-domain Gerchberg-Saxton loop on them and compares the recovered spectrum with the true one.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load('desk_capture.npz')
lam, c = 1550e-9, 299792458.0
b1, b2 = (-float(d[k])*1e-3*lam**2/(2*np.pi*c) for k in ('D1_psnm', 'D2_psnm'))
w = 2*np.pi*d['f']; H12 = np.exp(1j*(b2-b1)*w**2/2)
f1, f2 = np.sqrt(d['I1']), np.sqrt(d['I2']); u = f1.astype(complex); err = []
for _ in range(int(d['n_iter'])):
    u2 = np.fft.ifft(np.fft.fft(u)*H12); err.append(np.sqrt(np.mean((abs(u2)-f2)**2))/f2.max())
    u2 = f2*np.exp(1j*np.angle(u2)); u = f1*np.exp(1j*np.angle(np.fft.ifft(np.fft.fft(u2)/H12)))
U = np.fft.fft(u)*np.exp(-1j*b1*w**2/2); T = d['truth_re'] + 1j*d['truth_im']
U *= np.vdot(U, T)/np.vdot(U, U)
print('field error vs truth:', round(float(np.linalg.norm(U-T)/np.linalg.norm(T)), 3))

In [ ]:
fs = np.fft.fftshift(d['f'])/1e9
fig, ax = plt.subplots(2, 1, figsize=(7, 6))
ax[0].plot(fs, np.fft.fftshift(abs(T)), 'k', label='true')
ax[0].plot(fs, np.fft.fftshift(abs(U)), 'r--', label='recovered'); ax[0].set_xlim(-120, 120)
ax[0].set_xlabel('frequency offset (GHz)'); ax[0].legend()
ax[1].semilogy(err); ax[1].set_xlabel('iteration'); ax[1].set_ylabel('magnitude error')
plt.tight_layout(); plt.show()
print('Top: true (solid black) vs recovered (dashed red) spectrum after the gas cell. '
      'Bottom: TDGSA magnitude error per iteration on a log scale.')